# Tracefold News GEPA frozen-run evaluation

```yaml
channel: B  # frozen artifact: an operator-owned run directory on disk, never a database or a model endpoint
purpose: "Evaluate one completed Review -> Freeze -> Readiness -> Baseline -> GEPA -> Terminal Report run as a single internally consistent experiment, and say whether its candidate may proceed to independent evaluation."
window: "The closed window the frozen DevelopmentDataset already carries. This notebook takes no window of its own and reads no clock."
identity: "Read from the run and reconciled across its five files: development_dataset_sha, episode_projection_root_sha256, program_sha256/program_version, execution envelope_sha256, policy_sha256, stable bundle_sha, learning_epoch, review_rubric_version, metric_id. The run directory is named by TRACEFOLD_GEPA_RUN_ROOT, so the notebook pins no identity of its own."
safety: "Files only: no PostgreSQL connection, no model endpoint, no Review acceptance, no candidate registration, no promotion. Frozen datasets and evaluation reports are never repository content, so this notebook is committed with its outputs stripped."
```

**Where the run directory comes from.** `tracefold news learning freeze --role development --out
DIR/development.json` followed by `tracefold news learning run --development SHA --out DIR` (#253)
produces exactly the layout below, including `run_summary.json` — the operator-facing projection that
names the standalone and GEPA-seed baselines apart and states whether the two may be compared. Read
that first; this notebook is the long form of the same run, and it re-derives its own checks rather
than trusting the summary.

Repository security policy forbids committing frozen datasets and evaluation reports. Set the absolute path
to a frozen run before launching Jupyter; the tracked notebook intentionally contains no run data or outputs.
Every run must also carry an operator-owned `research-caveats.json`, even when its `items` list is empty.
This template requires Objective Plan v2, optimization objective summary v2, and split receipt v2; it fails
closed on historical v1 artifacts instead of comparing their Event-weighted results with representative-weighted runs.

```bash
uv sync --group research
TRACEFOLD_GEPA_RUN_ROOT=/absolute/path/to/frozen-run \
  uv run jupyter lab notebooks/news-gepa-frozen-run-evaluation.ipynb
```

Use **Restart Kernel and Run All Cells**. Historical filenames can be selected with
`TRACEFOLD_GEPA_DEVELOPMENT`, `TRACEFOLD_GEPA_READINESS`, and `TRACEFOLD_GEPA_CAVEATS`; otherwise the loader
requires exactly one `development*.json`, one `readiness*.json`, and one `research-caveats.json`. Save executed
copies only outside the repository.

## Setup

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display


PALETTE = {
    "ink": "#20242A",
    "muted": "#667085",
    "grid": "#D9DEE7",
    "blue": "#3568D4",
    "blue_light": "#C9D8F6",
    "gold": "#C28A18",
    "gold_light": "#F0DCA8",
    "orange": "#D9682B",
    "olive": "#73823B",
    "paper": "#FBFCFE",
}

plt.rcParams.update(
    {
        "figure.facecolor": PALETTE["paper"],
        "axes.facecolor": PALETTE["paper"],
        "axes.edgecolor": PALETTE["ink"],
        "axes.labelcolor": PALETTE["ink"],
        "axes.titlecolor": PALETTE["ink"],
        "text.color": PALETTE["ink"],
        "xtick.color": PALETTE["muted"],
        "ytick.color": PALETTE["ink"],
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "axes.grid": False,
        "figure.dpi": 120,
        "savefig.bbox": "tight",
    }
)

In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def resolve_input(run_root: Path, env_name: str, pattern: str) -> Path:
    override = os.environ.get(env_name, "").strip()
    if override:
        selected = Path(override).expanduser()
        selected = selected if selected.is_absolute() else run_root / selected
        selected = selected.resolve()
        if not selected.is_file():
            raise FileNotFoundError(f"{env_name}_not_found:{selected}")
        return selected
    matches = sorted(run_root.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(f"expected_one_{pattern}:found={len(matches)}")
    return matches[0].resolve()


run_root_text = os.environ.get("TRACEFOLD_GEPA_RUN_ROOT", "").strip()
if not run_root_text:
    raise RuntimeError("TRACEFOLD_GEPA_RUN_ROOT_must_name_an_operator_owned_frozen_run")

run_root = Path(run_root_text).expanduser().resolve()
if not run_root.is_dir():
    raise NotADirectoryError(f"frozen_run_root_not_found:{run_root}")

run_files = {
    "development": resolve_input(run_root, "TRACEFOLD_GEPA_DEVELOPMENT", "development*.json"),
    "readiness": resolve_input(run_root, "TRACEFOLD_GEPA_READINESS", "readiness*.json"),
    "baseline": (run_root / "baseline-compile-live.json").resolve(),
    "optimization": (run_root / "optimization/optimization_report.json").resolve(),
    "caveats": resolve_input(run_root, "TRACEFOLD_GEPA_CAVEATS", "research-caveats.json"),
}
missing = [str(path) for path in run_files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"missing_frozen_run_artifacts:{missing}")

artifacts = {name: read_json(path) for name, path in run_files.items()}
development = artifacts["development"]
readiness = artifacts["readiness"]
baseline = artifacts["baseline"]
optimization = artifacts["optimization"]
caveats = artifacts["caveats"]
prompt_candidate_path = run_root / "optimization/prompt_candidate.json"
prompt_candidate_file_present = prompt_candidate_path.is_file()
prompt_candidate = read_json(prompt_candidate_path) if prompt_candidate_file_present else None

provenance = {
    "source_kind": "operator-owned frozen run directory",
    "run_label": run_root.name,
    "files": {
        name: {"name": path.relative_to(run_root).as_posix(), "sha256": file_sha256(path)}
        for name, path in run_files.items()
    },
}

trajectory_scores = list(optimization.get("trajectory", {}).get("val_aggregate_scores") or [])
best_index = optimization.get("trajectory", {}).get("best_idx")
usage = optimization["usage"]
wall_seconds = (optimization["completed_at_ms"] - optimization["started_at_ms"]) / 1000
selection_score = baseline["subsets"]["development_selection"]["case_macro_failure_as_zero"]
stable_gepa_score = trajectory_scores[0] if trajectory_scores else None


def score_text(value: float | None) -> str:
    return "unavailable" if value is None else f"{value:.3f}"

## Data integrity gate

In [ ]:
checks = []


def check(name: str, condition: bool, detail: str) -> None:
    checks.append({"check": name, "ok": bool(condition), "detail": detail})


def canonical(value: object) -> str:
    return json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str, allow_nan=False
    )


def canonical_sha(value: object) -> str:
    return hashlib.sha256(canonical(value).encode()).hexdigest()


def derive_split_membership(
    cases: list[dict], optimizer_case_ids: list[str], split_receipt: dict
) -> tuple[set[str], set[str], list[str], list[str]]:
    by_case_id = {str(case["case_id"]): case for case in cases}
    representatives = [by_case_id[case_id] for case_id in optimizer_case_ids]
    representative_clusters = {str(case["cluster_id"]) for case in representatives}
    if len(representatives) != len(representative_clusters):
        raise AssertionError("optimizer_population_contains_duplicate_cluster")
    ordered_representatives = sorted(
        representatives, key=lambda case: (int(case["opened_at_ms"]), str(case["cluster_id"]))
    )
    share = float(split_receipt["policy"]["share"])
    cut = max(1, min(len(ordered_representatives) - 1, round(len(ordered_representatives) * share)))
    train = ordered_representatives[:cut]
    selection = ordered_representatives[cut:]
    train_ids = {str(case["case_id"]) for case in train}
    selection_ids = {str(case["case_id"]) for case in selection}
    train_clusters = [str(case["cluster_id"]) for case in train]
    selection_clusters = [str(case["cluster_id"]) for case in selection]
    return train_ids, selection_ids, train_clusters, selection_clusters


expected_schemas = {
    "development": "news_learning_dataset_v1",
    "readiness": "tracefold.news.gepa_readiness_report.v2",
    "baseline": "tracefold.news.program_baseline_report.v3",
    "optimization": "news_optimization_run_report_v1",
    "objective_plan": "tracefold.news.gepa_objective_plan.v2",
    "objective_summary": "tracefold.news.optimization_objective_summary.v2",
    "split": "tracefold.news.compile_split_receipt.v2",
    "caveats": "tracefold.research.gepa_caveats.v1",
}
check("Development schema", development.get("dataset_version") == expected_schemas["development"], str(development.get("dataset_version", "")))
check("Readiness schema", readiness.get("schema") == expected_schemas["readiness"], str(readiness.get("schema", "")))
check("Baseline schema", baseline.get("schema_id") == expected_schemas["baseline"], str(baseline.get("schema_id", "")))
check("Terminal schema", optimization.get("schema_version") == expected_schemas["optimization"], str(optimization.get("schema_version", "")))
objective_plan_schemas = {readiness["objective"].get("schema"), baseline["objective"].get("schema"), optimization["objective"].get("plan_schema")}
check("Objective Plan v2", objective_plan_schemas == {expected_schemas["objective_plan"]}, canonical(objective_plan_schemas))
check("Objective summary v2", optimization["objective"].get("schema") == expected_schemas["objective_summary"], str(optimization["objective"].get("schema", "")))
check("Split receipt v2", readiness["split"].get("schema") == expected_schemas["split"], str(readiness["split"].get("schema", "")))
check("Caveat-register schema", caveats.get("schema") == expected_schemas["caveats"], str(caveats.get("schema", "")))
caveat_items = caveats.get("items")
caveats_valid = isinstance(caveat_items, list) and all(isinstance(item, str) and item.strip() for item in caveat_items)
check("Caveat register reviewed", caveats_valid, f"items={len(caveat_items) if isinstance(caveat_items, list) else 'invalid'}")

development_address = str(development.get("artifact_sha") or "")
development_payload = {key: value for key, value in development.items() if key != "artifact_sha"}
development_hash_ok = development_address == canonical_sha({"kind": "dataset", "payload": development_payload})
check("DevelopmentDataset self-hash", development_hash_ok, f"artifact={development_address[:12]}…")

baseline_address = str(baseline.get("report_sha256") or "")
baseline_measurement = {key: value for key, value in baseline.items() if key != "report_sha256"}
baseline_measurement["latency_ms"] = {}
baseline_measurement["cases"] = [{**case, "latency_ms": 0} for case in baseline_measurement["cases"]]
baseline_hash_ok = baseline_address == canonical_sha(baseline_measurement)
check("Baseline measurement self-hash", baseline_hash_ok, f"report={baseline_address[:12]}…")

optimization_address = str(optimization.get("report_sha256") or "")
optimization_payload = {key: value for key, value in optimization.items() if key != "report_sha256"}
optimization_hash_ok = optimization_address == canonical_sha(optimization_payload)
check("Terminal report self-hash", optimization_hash_ok, f"report={optimization_address[:12]}…")

dataset_shas = {
    development["artifact_sha"],
    readiness["identity"]["development_dataset_sha"],
    baseline["identity"]["development_dataset_sha"],
    optimization["dataset"]["development_dataset_sha256"],
}
episode_roots = {
    readiness["identity"]["episode_projection_root_sha256"],
    baseline["identity"]["episode_projection_root_sha256"],
    optimization["dataset"]["episode_projection_root_sha256"],
    optimization["objective"]["episode_projection_root_sha256"],
}
program_shas = {
    development["agent_cohort"]["program_sha256"],
    readiness["identity"]["program_sha256"],
    baseline["identity"]["program_sha256"],
    optimization["parent_program_sha256"],
}
program_versions = {
    development["agent_cohort"]["program_version"],
    readiness["identity"]["program_version"],
    baseline["identity"]["program_version"],
}
policy_shas = {
    development["agent_cohort"]["policy_sha256"],
    readiness["identity"]["policy_sha256"],
    baseline["identity"]["policy_sha256"],
}
envelope_shas = {readiness["identity"]["execution_envelope_sha256"], baseline["identity"]["envelope_sha256"]}
stable_bundles = {development["agent_cohort"]["bundle_sha"], readiness["identity"]["stable_bundle_sha"]}
learning_epochs = {
    development["agent_cohort"]["learning_epoch"],
    readiness["identity"]["learning_epoch"],
    optimization["dataset"]["learning_epoch"],
}
rubrics = set(development["counts"]["eligibility"]["rubric_versions"]) | {
    readiness["identity"]["review_rubric_version"],
    optimization["dataset"]["review_rubric_version"],
}
metric_ids = {baseline["identity"]["metric_id"], optimization["metric"]["metric_id"]}
runtime_manifests = {
    development["agent_cohort"]["runtime_model_bindings_sha256"],
    optimization["target_runtime_manifest_sha256"],
}
task_models = {
    readiness["identity"]["model_targets"]["task"]["model"],
    optimization["model_identities"]["task"]["model"],
}
judge_models = {
    baseline["identity"]["metric"]["semantic_judge"]["model"],
    optimization["model_identities"]["metric_judge"]["model"],
}

for label, values in (
    ("One dataset identity", dataset_shas),
    ("One episode projection", episode_roots),
    ("One Program SHA", program_shas),
    ("One Program version", program_versions),
    ("One policy SHA", policy_shas),
    ("One execution envelope", envelope_shas),
    ("One Stable bundle", stable_bundles),
    ("One learning epoch", learning_epochs),
    ("One Review rubric", rubrics),
    ("One metric identity", metric_ids),
    ("One target runtime manifest", runtime_manifests),
    ("One task model", task_models),
    ("One metric-judge model", judge_models),
):
    check(label, len(values) == 1 and None not in values and "" not in values, f"unique values={len(values)}")

required_model_roles = ("task", "reflection", "metric_judge")
model_roles_complete = all(
    optimization.get("model_identities", {}).get(role, {}).get("model") for role in required_model_roles
)
check("All model roles identified", model_roles_complete, ", ".join(required_model_roles))

case_count = int(development["counts"]["case_n"])
cluster_ids = {case["cluster_id"] for case in development["cases"]}
case_ids = {case["case_id"] for case in development["cases"]}
check(
    "Case count reconciles",
    case_count == len(development["cases"]) == readiness["corpus"]["case_n"] == optimization["dataset"]["episode_count"],
    f"case_n={case_count}",
)
check("Case IDs are unique", len(case_ids) == case_count, f"unique={len(case_ids)}")
optimizer_case_ids = {str(case_id) for case_id in optimization["objective"]["optimizer_case_ids"]}
optimizer_cluster_ids = {str(case["cluster_id"]) for case in development["cases"] if str(case["case_id"]) in optimizer_case_ids}
baseline_case_ids = {str(case["case_id"]) for case in baseline["cases"]}
check("Baseline scores representative set", baseline_case_ids == optimizer_case_ids, f"baseline={len(baseline_case_ids)}, optimizer={len(optimizer_case_ids)}")
check("Readiness dispositions cover full Dataset", len(readiness["case_dispositions"]) == case_count, f"dispositions={len(readiness['case_dispositions'])}, dataset={case_count}")
check("Cluster grain reconciles", len(cluster_ids) == readiness["corpus"]["cluster_n"] == optimization["objective"]["cluster_n"], f"unique clusters={len(cluster_ids)}")

readiness_objective = {
    "target_case_n": readiness["objective"]["target_case_n"],
    "control_case_n": readiness["objective"]["control_case_n"],
    "excluded_case_n": readiness["objective"]["excluded_case_n"],
    "target_cluster_n": readiness["objective"]["target_cluster_n"],
    "control_cluster_n": readiness["objective"]["control_cluster_n"],
    "optimizer_case_n": readiness["objective"]["optimizer_case_n"],
    "optimizer_cluster_n": readiness["objective"]["optimizer_cluster_n"],
    "optimizer_case_root_sha256": readiness["objective"]["optimizer_case_root_sha256"],
    "exclusion_reasons": readiness["objective"]["exclusion_reasons"],
    "target_dimensions": sorted(readiness["objective"]["target_dimensions"]),
    "target_failure_cluster_ids": sorted(readiness["objective"]["target_failure_cluster_ids"]),
    "target_predictors": sorted(readiness["objective"]["target_predictors"]),
}
baseline_objective = {
    key: sorted(baseline["objective"][key]) if key in {"target_dimensions", "target_failure_cluster_ids", "target_predictors"} else baseline["objective"][key]
    for key in readiness_objective
}
optimization_objective = {
    "target_case_n": len(optimization["objective"]["target_case_ids"]),
    "control_case_n": len(optimization["objective"]["control_case_ids"]),
    "excluded_case_n": len(optimization["objective"]["excluded_case_ids"]),
    "target_cluster_n": len(optimization["objective"]["target_failure_cluster_ids"]),
    "control_cluster_n": len(optimization["objective"]["control_cluster_ids"]),
    "optimizer_case_n": optimization["objective"]["optimizer_case_n"],
    "optimizer_cluster_n": optimization["objective"]["optimizer_cluster_n"],
    "optimizer_case_root_sha256": optimization["objective"]["optimizer_case_root_sha256"],
    "exclusion_reasons": optimization["objective"]["exclusion_reasons"],
    "target_dimensions": sorted(optimization["objective"]["target_dimensions"]),
    "target_failure_cluster_ids": sorted(optimization["objective"]["target_failure_cluster_ids"]),
    "target_predictors": sorted(optimization["objective"]["target_predictors"]),
}
objective_receipts = {canonical(item) for item in (readiness_objective, baseline_objective, optimization_objective)}
check("Objective receipts reconcile", len(objective_receipts) == 1, f"unique receipts={len(objective_receipts)}")

objective_total = sum(readiness_objective[name] for name in ("target_case_n", "control_case_n", "excluded_case_n"))
optimizer_total = readiness_objective["target_case_n"] + readiness_objective["control_case_n"]
check("Disposition accounting", objective_total == case_count, f"target+control+excluded={objective_total}")
optimizer_identity_ok = (
    optimizer_total == len(optimizer_case_ids) == len(optimizer_cluster_ids)
    and optimizer_total == readiness["objective"]["optimizer_case_n"] == baseline["objective"]["optimizer_case_n"] == optimization["objective"]["optimizer_case_n"]
    and len(optimizer_cluster_ids) == readiness["objective"]["optimizer_cluster_n"] == baseline["objective"]["optimizer_cluster_n"] == optimization["objective"]["optimizer_cluster_n"]
    and canonical_sha(sorted(optimizer_case_ids)) == readiness["objective"]["optimizer_case_root_sha256"] == baseline["objective"]["optimizer_case_root_sha256"] == optimization["objective"]["optimizer_case_root_sha256"]
)
check("One representative per optimizer cluster", optimizer_identity_ok, f"cases={len(optimizer_case_ids)}, clusters={len(optimizer_cluster_ids)}")
check("Terminal full-corpus accounting", optimization["objective"]["case_n"] == case_count, f"objective case_n={optimization['objective']['case_n']}")

split_sources = [readiness["split"], baseline["objective"]["split"], optimization["objective"]["split"], optimization["split"]]
check("Complete split receipts match", len({canonical(item) for item in split_sources}) == 1, "readiness=baseline=objective=terminal")

split_policy = readiness["split"]["policy"]
split_policy_supported = (
    split_policy.get("unit") == "connected_fact_cluster_representative"
    and split_policy.get("representative_n_per_cluster") == 1
    and split_policy.get("representative_order") == ["target_before_control", "target_dimension_n_desc", "safety_strength_desc", "event_time_desc", "case_id"]
    and split_policy.get("split_order") == ["event_time", "cluster_id"]
    and 0 < float(split_policy.get("share", 0)) < 1
)
check("Split policy is reproducible", split_policy_supported, canonical(split_policy))
train_case_ids, selection_case_ids, train_cluster_ids, selection_cluster_ids = derive_split_membership(
    development["cases"], list(optimization["objective"]["optimizer_case_ids"]), readiness["split"]
)
derived_split_roots = {
    "train": {
        "case_n": len(train_case_ids),
        "cluster_n": len(train_cluster_ids),
        "case_root_sha256": canonical_sha(sorted(train_case_ids)),
        "cluster_root_sha256": canonical_sha(train_cluster_ids),
    },
    "development_selection": {
        "case_n": len(selection_case_ids),
        "cluster_n": len(selection_cluster_ids),
        "case_root_sha256": canonical_sha(sorted(selection_case_ids)),
        "cluster_root_sha256": canonical_sha(selection_cluster_ids),
    },
}
split_membership_ok = all(
    all(readiness["split"][side][field] == value for field, value in derived_split_roots[side].items())
    for side in ("train", "development_selection")
)
check("Split membership roots re-derived", split_membership_ok, "case and cluster roots match frozen cases")

selection_hard_gates: dict[str, int] = {}
for case in baseline["cases"]:
    gate = str(case.get("hard_gate") or "")
    if str(case["case_id"]) in selection_case_ids and gate:
        selection_hard_gates[gate] = selection_hard_gates.get(gate, 0) + 1
selection_hard_gate_n = sum(selection_hard_gates.values())
check(
    "Selection hard gates reconcile",
    selection_hard_gate_n == baseline["subsets"]["development_selection"]["hard_gate_n"],
    f"derived={selection_hard_gate_n}",
)
for side in ("train", "development_selection"):
    split_side = readiness["split"][side]
    readiness_side = readiness[side]
    counts_match = (
        split_side["case_n"] == readiness_side["case_n"] == baseline["subsets"][side]["case_n"]
        and split_side["cluster_n"] == readiness_side["cluster_n"] == baseline["subsets"][side]["cluster_n"]
    )
    coverage_matches = split_side["coverage"] == readiness_side["strata"]
    check(f"{side} counts match", counts_match, f"cases={split_side['case_n']}, clusters={split_side['cluster_n']}")
    check(f"{side} coverage matches", coverage_matches, canonical(split_side["coverage"]))
check(
    "Split is disjoint",
    readiness["split"]["disjointness"]["shared_case_ids"] == 0 and readiness["split"]["disjointness"]["shared_clusters"] == 0,
    "shared cases=0, clusters=0",
)

settlement_lag = development["freeze_as_of_ms"] - development["window"]["to_ms"]
check("Settlement grace respected", settlement_lag >= development["settlement_grace_ms"], f"lag={settlement_lag:,} ms")
check("Readiness has no blockers", readiness["outcome"] == "ready" and not readiness["blocking_reasons"], str(readiness["outcome"]))
check("Baseline answered every representative", baseline["population"]["failure_n"] == 0 and baseline["population"]["answered_n"] == len(optimizer_case_ids), f"answered={baseline['population']['answered_n']}")
check("Known terminal outcome", optimization["outcome"] in {"ADVANCE", "NO_OP", "REJECTED"}, str(optimization["outcome"]))
check("GEPA actually ran", usage["task_model_calls"] > 0 and usage["reflection_model_calls"] > 0, f"task={usage['task_model_calls']}, reflection={usage['reflection_model_calls']}")

candidate_present = bool(optimization.get("candidate_sha256"))
candidate_file_present = prompt_candidate_file_present
candidate_contract_ok = (
    candidate_present and candidate_file_present
    if optimization["outcome"] == "ADVANCE"
    else not candidate_present and not candidate_file_present
)
check("Candidate matches terminal state", candidate_contract_ok, f"outcome={optimization['outcome']}, receipt={candidate_present}, file={candidate_file_present}")

candidate_hash_ok = optimization["outcome"] != "ADVANCE"
candidate_schema_ok = optimization["outcome"] != "ADVANCE"
candidate_identity_ok = optimization["outcome"] != "ADVANCE"
if optimization["outcome"] == "ADVANCE" and isinstance(prompt_candidate, dict):
    candidate_fields = {
        "schema_version", "parent_program_sha256", "development_dataset_sha256",
        "target_runtime_manifest_sha256", "patch", "objective_summary", "optimizer",
        "model_identities", "budget", "usage", "created_at_ms", "candidate_sha256",
    }
    patch = prompt_candidate.get("patch")
    candidate_schema_ok = (
        set(prompt_candidate) == candidate_fields
        and isinstance(patch, dict)
        and set(patch) == {"event_semantics_instruction", "reader_card_instruction"}
        and all(isinstance(patch[name], str) for name in patch)
        and all(isinstance(prompt_candidate.get(name), dict) for name in ("objective_summary", "optimizer", "model_identities", "budget", "usage"))
        and isinstance(prompt_candidate.get("created_at_ms"), int)
        and prompt_candidate["created_at_ms"] >= 0
    )
    candidate_address = str(prompt_candidate.get("candidate_sha256") or "")
    candidate_payload = {key: value for key, value in prompt_candidate.items() if key != "candidate_sha256"}
    candidate_hash_ok = (
        candidate_address == canonical_sha(candidate_payload) == str(optimization["candidate_sha256"])
    )
    candidate_objective = prompt_candidate.get("objective_summary") or {}
    candidate_identity_ok = (
        prompt_candidate.get("schema_version") == "news_prompt_candidate_v1"
        and prompt_candidate.get("parent_program_sha256") == optimization["parent_program_sha256"]
        and prompt_candidate.get("development_dataset_sha256") == optimization["dataset"]["development_dataset_sha256"]
        and prompt_candidate.get("target_runtime_manifest_sha256") == optimization["target_runtime_manifest_sha256"]
        and candidate_objective.get("schema") == expected_schemas["objective_summary"]
        and candidate_objective.get("plan_schema") == expected_schemas["objective_plan"]
        and candidate_objective.get("optimizer_case_ids") == optimization["objective"]["optimizer_case_ids"]
        and candidate_objective.get("optimizer_case_root_sha256") == optimization["objective"]["optimizer_case_root_sha256"]
    )
check("Candidate schema is complete", candidate_schema_ok, "not applicable" if not candidate_present else "exact v1 fields and Prompt-only patch")
check("Candidate self-hash matches terminal receipt", candidate_hash_ok, "not applicable" if not candidate_present else "candidate and terminal SHA agree")
check("Candidate identities match terminal report", candidate_identity_ok, "not applicable" if not candidate_present else "schema, Program, Dataset, runtime agree")

if best_index is not None:
    best_index_valid = isinstance(best_index, int) and 0 <= best_index < len(trajectory_scores)
    best_score_valid = best_index_valid and trajectory_scores[best_index] == max(trajectory_scores)
else:
    best_index_valid = not trajectory_scores
    best_score_valid = not trajectory_scores
check("Trajectory best index is valid", best_index_valid, f"scores={len(trajectory_scores)}, best_idx={best_index}")
check("Trajectory best score reconciles", best_score_valid, f"best_idx={best_index}")

failed_checks = [item for item in checks if not item["ok"]]
if failed_checks:
    evidence = "; ".join(f"{item['check']} ({item['detail']})" for item in failed_checks)
    raise AssertionError(f"frozen_run_validation_failed:{evidence}")

rows = ["| Check | Status | Evidence |", "|---|:---:|---|"]
for item in checks:
    rows.append(f"| {item['check']} | PASS | {item['detail']} |")
display(Markdown("\n".join(rows)))

In [ ]:
window_start = datetime.fromtimestamp(development["window"]["from_ms"] / 1000, tz=UTC)
window_end = datetime.fromtimestamp(development["window"]["to_ms"] / 1000, tz=UTC)
freeze_time = datetime.fromtimestamp(development["freeze_as_of_ms"] / 1000, tz=UTC)

file_rows = "\n".join(
    f"| {name} | `{item['name']}` | `{item['sha256']}` |" for name, item in provenance["files"].items()
)
data_note = f'''
### Frozen corpus identity

| Field | Value |
|---|---|
| Source | `{provenance['source_kind']}` / `{provenance['run_label']}` |
| Window (UTC) | `{window_start.isoformat()}` → `{window_end.isoformat()}` |
| Frozen at (UTC) | `{freeze_time.isoformat()}` |
| DevelopmentDataset | `{development['artifact_sha']}` |
| Episode projection | `{readiness['identity']['episode_projection_root_sha256']}` |
| Program | `{readiness['identity']['program_sha256']}` |
| Policy | `{readiness['identity']['policy_sha256']}` |
| Review rubric | `{readiness['identity']['review_rubric_version']}` |
| Metric | `{baseline['identity']['metric_id']}` |

### File provenance

| Artifact | Relative path | File SHA-256 |
|---|---|---|
{file_rows}
'''
display(Markdown(data_note))

## tl;dr

In [ ]:
outcome = optimization["outcome"]
selection_n = readiness["development_selection"]["case_n"]
hard_gate_n = baseline["subsets"]["development_selection"]["hard_gate_n"]

if outcome == "ADVANCE":
    assessment = "Candidate available; independent release evaluation required"
    terminal_decision = "Do not promote automatically. Move the emitted candidate to the independent holdout and release-evidence workflow."
elif outcome == "NO_OP":
    assessment = "Stable retained; evidence is diagnostic"
    terminal_decision = "Keep Stable for this corpus."
else:
    assessment = "Run rejected; evidence needs investigation"
    reasons = ", ".join(optimization.get("reasons") or ["no reason recorded"])
    terminal_decision = f"Keep Stable and investigate the terminal reasons: {reasons}."

if trajectory_scores:
    selection_result = (
        f"GEPA evaluated {len(trajectory_scores)} trajectory entr{'y' if len(trajectory_scores) == 1 else 'ies'}; "
        f"best index **{best_index}** scored **{score_text(trajectory_scores[best_index])}**, "
        f"versus Stable seed **{score_text(stable_gepa_score)}**."
    )
else:
    selection_result = "The terminal report contains no scored trajectory entries."

caveat_lines = "\n".join(f"- {item.strip()}" for item in caveat_items)
if not caveat_lines:
    caveat_lines = "- The operator-reviewed caveat register declares no additional run-specific limitations."

assessment_caveat = (
    f"The selection side has only **{selection_n} cases** and **{hard_gate_n} hard-gated cases**; "
    "this is not a production-volume estimator or release holdout."
)

summary = f'''
### Overall assessment: **{assessment}**

- The frozen corpus contains **{readiness['corpus']['case_n']} cases / {readiness['corpus']['cluster_n']} independent clusters**:
  **{readiness['objective']['target_case_n']} targets**, **{readiness['objective']['control_case_n']} controls**,
  and **{readiness['objective']['excluded_case_n']} excluded diagnostics**.
- Readiness is **`{readiness['outcome']}`** with {len(readiness['blocking_reasons'])} blockers; train and
  development-selection share **{readiness['split']['disjointness']['shared_clusters']} clusters**.
- Stable's standalone development-selection failure-as-zero score is **{selection_score:.3f}**.
- {selection_result}
- GEPA ended **`{outcome}`** and used **{usage['task_model_calls']} task**, **{usage['reflection_model_calls']} reflection**,
  **{usage['metric_judge_model_calls']} judge**, and **{usage['metric_calls']} metric** calls. Accounted cost was
  **${usage['actual_cost_microusd'] / 1_000_000:.2f}** and wall time **{wall_seconds:.1f} s**.

**Decision:** {terminal_decision} {assessment_caveat}

**Operator-declared run caveats**

{caveat_lines}
'''
display(Markdown(summary))

## Context & Methods

### Analytical question

Does this frozen run form one internally consistent experiment, where did Stable fail on the reviewed evidence,
and did bounded GEPA produce a candidate that may proceed to independent evaluation?

### Method

1. Reconcile schemas; Dataset, Episode, Program, policy, execution-envelope, metric, model-runtime and rubric identities;
   objective counts; all split receipts; the required caveat register; and candidate/terminal state before any
   analytical output.
2. Reconcile the Objective Plan v2 representative set, then describe corpus composition and disjoint train/development-selection coverage at connected-fact-cluster grain.
3. Evaluate Stable using failure-as-zero subset scores, hard gates, exact-gold coverage and dimension outcomes.
4. Compare every recorded GEPA trajectory entry and each typed call/cost/time ceiling.
5. Interpret the terminal state without treating `ADVANCE` as promotion or `NO_OP` as global optimality.

### Limits

- PostgreSQL material facts and accepted Reviews remain business truth; frozen files are immutable experiment evidence.
- `development_selection` selects a candidate; it is not a future temporal holdout.
- A deliberately selected, small corpus does not support confidence intervals, causal claims, or a production
  push-volume estimate. The cluster count—not the raw case count—is the independence unit.
- Price reaction is not a quality label and is intentionally absent.

### Chart map

| Question | Visual | Why this form |
|---|---|---|
| Is the corpus usable? | composition + grouped split bars | exact counts and side-by-side coverage |
| Where is Stable weak? | subset scores + component gold coverage | bounded 0–1 comparisons with denominators |
| Which dimensions fail? | ranked horizontal lollipop | long labels and exact sample sizes |
| Did GEPA improve within bounds? | trajectory bars + budget utilization | discrete candidates and typed ceilings |

## Results

In [ ]:
# Visual 1 — corpus composition and split coverage.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [0.85, 1.55]})

composition = {
    "Target": readiness["objective"]["target_case_n"],
    "Control": readiness["objective"]["control_case_n"],
    "Excluded": readiness["objective"]["excluded_case_n"],
}
colors = [PALETTE["orange"], PALETTE["blue"], PALETTE["gold_light"]]
left = 0
for (label, value), color in zip(composition.items(), colors, strict=True):
    axes[0].barh([0], [value], left=left, color=color, edgecolor=PALETTE["ink"], linewidth=0.8, label=label)
    if value:
        label_color = "white" if label != "Excluded" else PALETTE["ink"]
        axes[0].text(left + value / 2, 0, str(value), ha="center", va="center", color=label_color, fontweight="bold")
    left += value
axes[0].set_xlim(0, max(1, sum(composition.values())))
axes[0].set_yticks([])
axes[0].set_xlabel("Cases")
axes[0].set_title("Corpus disposition")
axes[0].legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.38), ncol=3)
axes[0].spines[["top", "right", "left"]].set_visible(False)

coverage_names = ["target", "control", "positive", "negative", "safety", "novelty"]
train_values = [
    readiness["train"]["target_case_n"],
    readiness["train"]["control_case_n"],
    readiness["train"]["strata"]["positive_action"],
    readiness["train"]["strata"]["negative_action"],
    readiness["train"]["strata"]["safety"],
    readiness["train"]["strata"]["novelty"],
]
selection_values = [
    readiness["development_selection"]["target_case_n"],
    readiness["development_selection"]["control_case_n"],
    readiness["development_selection"]["strata"]["positive_action"],
    readiness["development_selection"]["strata"]["negative_action"],
    readiness["development_selection"]["strata"]["safety"],
    readiness["development_selection"]["strata"]["novelty"],
]
x = list(range(len(coverage_names)))
width = 0.36
axes[1].bar([i - width / 2 for i in x], train_values, width, color=PALETTE["blue"], edgecolor=PALETTE["ink"], linewidth=0.7, label=f"Train (n={readiness['train']['case_n']})")
axes[1].bar([i + width / 2 for i in x], selection_values, width, color=PALETTE["gold"], edgecolor=PALETTE["ink"], linewidth=0.7, label=f"Selection (n={readiness['development_selection']['case_n']})")
for offset, values in ((-width / 2, train_values), (width / 2, selection_values)):
    for index, value in enumerate(values):
        axes[1].text(index + offset, value + 0.08, str(value), ha="center", va="bottom", fontsize=9)
axes[1].set_xticks(x, coverage_names, rotation=22, ha="right")
axes[1].set_ylabel("Cases carrying coverage")
axes[1].set_ylim(0, max(train_values + selection_values) + 1)
axes[1].set_title("Coverage across the disjoint split")
axes[1].yaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
axes[1].set_axisbelow(True)
axes[1].legend(frameon=False, loc="upper left")
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Frozen corpus readiness", x=0.06, ha="left", fontsize=16, fontweight="bold")
fig.text(0.06, 0.92, f"n={case_count} frozen cases; optimizer n={len(optimizer_case_ids)} representatives; shared clusters=0", color=PALETTE["muted"])
plt.tight_layout(rect=(0, 0.04, 1, 0.88))
plt.show()

In [ ]:
# Visual 2 — baseline scores and how much exact gold supports each component.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

subset_order = ["train", "development_selection", "optimizer_union"]
subset_labels = ["Train", "Development selection", "Optimizer union"]
subset_scores = [baseline["subsets"][name]["case_macro_failure_as_zero"] for name in subset_order]
subset_ns = [baseline["subsets"][name]["case_n"] for name in subset_order]
subset_gates = [baseline["subsets"][name]["hard_gate_n"] for name in subset_order]
y = list(range(len(subset_order)))
axes[0].barh(y, subset_scores, color=[PALETTE["blue"], PALETTE["orange"], PALETTE["blue_light"]], edgecolor=PALETTE["ink"], linewidth=0.8)
for index, (score, n, gates) in enumerate(zip(subset_scores, subset_ns, subset_gates, strict=True)):
    axes[0].text(max(score + 0.025, 0.025), index, f"{score:.3f}  · n={n}, hard gates={gates}", va="center", fontsize=9)
axes[0].set_yticks(y, subset_labels)
axes[0].invert_yaxis()
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Case macro failure-as-zero score (0–1)")
axes[0].set_title("Stable score by subset")
axes[0].xaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
axes[0].set_axisbelow(True)
axes[0].spines[["top", "right"]].set_visible(False)

components = ["final_action", "trade_relevance", "semantics_novelty", "reader_card"]
component_labels = ["Final action", "Trade relevance", "Semantics + novelty", "Reader card"]
diagnostics = baseline["scores"]["component_diagnostics"]
coverage = [diagnostics[name]["gold_coverage"] for name in components]
denominators = [diagnostics[name]["denominator"] for name in components]
y2 = list(range(len(components)))
axes[1].barh(y2, coverage, color=PALETTE["gold"], edgecolor=PALETTE["ink"], linewidth=0.8)
for index, (rate, denominator) in enumerate(zip(coverage, denominators, strict=True)):
    axes[1].text(min(rate + 0.025, 0.93), index, f"{rate:.1%}  · denominator={denominator}", va="center", fontsize=9)
axes[1].set_yticks(y2, component_labels)
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Exact-gold coverage")
axes[1].set_title("Metric evidence strength")
axes[1].xaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
axes[1].set_axisbelow(True)
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Stable Program baseline", x=0.06, ha="left", fontsize=16, fontweight="bold")
fig.text(0.06, 0.92, "Selection is the formal before value; exact-gold coverage shows how much scoring rests on stated correct values.", color=PALETTE["muted"])
plt.tight_layout(rect=(0, 0.02, 1, 0.88))
plt.show()

In [ ]:
# Visual 3 — dimension-level baseline behavior, ranked by hit rate.
dimension_rows = [
    (name, values["hit_rate"], values["n"], values.get("gold_hit", 0), values.get("gold_miss", 0))
    for name, values in baseline["prediction_dimensions"].items()
]
dimension_rows.sort(key=lambda item: (item[1], item[0]))

fig, ax = plt.subplots(figsize=(10.5, max(4.5, 0.48 * len(dimension_rows) + 2)))
if dimension_rows:
    y = list(range(len(dimension_rows)))
    rates = [row[1] for row in dimension_rows]
    ax.hlines(y, 0, rates, color=PALETTE["blue_light"], linewidth=5)
    ax.scatter(rates, y, s=68, color=PALETTE["blue"], edgecolor=PALETTE["ink"], linewidth=0.8, zorder=3)
    for index, (_, rate, n, gold_hit, gold_miss) in enumerate(dimension_rows):
        ax.text(min(rate + 0.025, 0.9), index, f"{rate:.1%} · n={n} · exact gold {gold_hit}/{gold_hit + gold_miss}", va="center", fontsize=9)
    ax.set_yticks(y, [row[0] for row in dimension_rows])
else:
    ax.text(0.5, 0.5, "No labelled dimension outcomes", ha="center", va="center", transform=ax.transAxes)
    ax.set_yticks([])
ax.set_xlim(0, 1.28)
ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_xlabel("Observed hit rate")
ax.set_title("Baseline dimension outcomes")
ax.xaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.125, 0.92, "n is the labelled denominator; exact-gold counts distinguish stated answers from retention checks.", color=PALETTE["muted"])
plt.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

In [ ]:
# Visual 4 — 0/1/N trajectory entries and typed budget utilization.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), gridspec_kw={"width_ratios": [0.9, 1.4]})

if trajectory_scores:
    candidate_labels = ["Stable seed" if index == 0 else f"Proposal {index}" for index in range(len(trajectory_scores))]
    candidate_colors = [PALETTE["blue"] if index == best_index else PALETTE["orange"] for index in range(len(trajectory_scores))]
    axes[0].bar(candidate_labels, trajectory_scores, color=candidate_colors, edgecolor=PALETTE["ink"], linewidth=0.8)
    for index, value in enumerate(trajectory_scores):
        axes[0].text(index, value + 0.025, f"{value:.3f}", ha="center", va="bottom", fontweight="bold")
    axes[0].tick_params(axis="x", rotation=25)
else:
    axes[0].text(0.5, 0.5, "No scored trajectory entries", ha="center", va="center", transform=axes[0].transAxes)
    axes[0].set_xticks([])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("GEPA validation aggregate score")
axes[0].set_title("Candidate selection")
axes[0].yaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
axes[0].set_axisbelow(True)
axes[0].spines[["top", "right"]].set_visible(False)

budget_rows = [
    ("Metric calls", usage["metric_calls"], optimization["budget"]["max_metric_calls"]),
    ("Task calls", usage["task_model_calls"], optimization["budget"]["max_task_model_calls"]),
    ("Reflection calls", usage["reflection_model_calls"], optimization["budget"]["max_reflection_model_calls"]),
    ("Judge calls", usage["metric_judge_model_calls"], optimization["budget"]["max_metric_judge_model_calls"]),
    ("Cost", usage["actual_cost_microusd"], optimization["budget"]["max_cost_microusd"]),
    ("Wall clock", wall_seconds, optimization["budget"]["max_wall_clock_seconds"]),
]
budget_rates = [actual / limit if limit else 0 for _, actual, limit in budget_rows]
y = list(range(len(budget_rows)))
axes[1].barh(y, [1] * len(y), color="#E9EDF3", edgecolor=PALETTE["grid"], linewidth=0.8)
axes[1].barh(y, budget_rates, color=PALETTE["gold"], edgecolor=PALETTE["ink"], linewidth=0.8)
for index, ((name, actual, limit), rate) in enumerate(zip(budget_rows, budget_rates, strict=True)):
    if name == "Cost":
        exact = f"${actual / 1_000_000:.2f} / ${limit / 1_000_000:.2f}"
    elif name == "Wall clock":
        exact = f"{actual:.1f}s / {limit:.0f}s"
    else:
        exact = f"{actual:g} / {limit:g}"
    axes[1].text(min(rate + 0.02, max(0.79, max(budget_rates) * 0.78)), index, f"{rate:.1%} · {exact}", va="center", fontsize=9)
axes[1].set_yticks(y, [row[0] for row in budget_rows])
axes[1].invert_yaxis()
axes[1].set_xlim(0, max(1, max(budget_rates, default=0) * 1.25))
axes[1].set_xlabel("Share of declared ceiling")
axes[1].set_title("Budget utilization")
axes[1].xaxis.grid(True, color=PALETTE["grid"], linewidth=0.7)
axes[1].set_axisbelow(True)
axes[1].spines[["top", "right"]].set_visible(False)

candidate_note = "candidate emitted" if candidate_present else "no candidate emitted"
fig.suptitle("GEPA terminal result", x=0.06, ha="left", fontsize=16, fontweight="bold")
fig.text(0.06, 0.92, f"Outcome={outcome}; best_idx={best_index}; {candidate_note}; seed={optimization['budget']['seed']}", color=PALETTE["muted"])
plt.tight_layout(rect=(0, 0.02, 1, 0.88))
plt.show()

## Takeaways

In [ ]:
hard_gates = selection_hard_gates
gold_rate = baseline["gold_coverage"]["rate"]
metric_limit = optimization["budget"]["max_metric_calls"]
metric_budget_rate = usage["metric_calls"] / metric_limit if metric_limit else 0
baseline_gap = None if stable_gepa_score is None else stable_gepa_score - selection_score
hard_gate_text = ", ".join(f"{name}={count}" for name, count in hard_gates.items()) or "none"

if trajectory_scores:
    selection_takeaway = (
        f"GEPA evaluated **{len(trajectory_scores)}** trajectory entries. Index **{best_index}** was best at "
        f"**{score_text(trajectory_scores[best_index])}**; Stable seed scored **{score_text(stable_gepa_score)}**."
    )
else:
    selection_takeaway = "The terminal report recorded no scored trajectory entries."

if outcome == "ADVANCE":
    terminal_takeaway = "`ADVANCE` emitted a candidate, but this is only permission for independent holdout and release evaluation—not promotion."
elif outcome == "NO_OP":
    terminal_takeaway = "`NO_OP` retains Stable for this frozen corpus; it does not prove the Program is globally optimal."
else:
    terminal_takeaway = "`REJECTED` retains Stable; diagnose the typed terminal reasons before rerunning the optimizer."

repeatability_note = (
    "No GEPA Stable re-evaluation score is available."
    if baseline_gap is None
    else f"Standalone selection and GEPA Stable differ by **{abs(baseline_gap):.3f}**; on a small live sample this is a repeatability warning, not uplift."
)

next_experiment = (
    "Send the candidate to an independent future holdout and the existing release gates; do not reuse this selection split."
    if outcome == "ADVANCE"
    else "Freeze a broader, multi-day current-cohort dataset with representative target and control clusters, then rerun readiness and baseline before spending another GEPA budget."
)

takeaways = f'''
### What the evidence supports

1. **The governed loop is internally consistent.** All identity families, Objective Plan v2 representative roots and split receipts agree;
   every optimizer representative answered in Baseline; GEPA used real task/reflection calls and wrote a typed terminal report.
2. **Candidate selection behaved according to its receipts.** {selection_takeaway} {terminal_takeaway}
3. **The corpus is diagnostic, not release-grade.** It has {case_count} cases / {len(cluster_ids)} independent clusters;
   development-selection has {readiness['development_selection']['case_n']} cases, with selection-only hard gates: {hard_gate_text}.
4. **Metric strength is visible.** Overall exact-gold coverage is **{gold_rate:.1%}**; component and dimension charts
   separate stated correct values from retention/judge evidence.
5. **Resource use is bounded and auditable.** Metric-call utilization was **{metric_budget_rate:.0%}**; call, judge,
   cost and wall-clock ceilings are reported separately.

### Required caveats

- {repeatability_note}
- This selected corpus cannot estimate how many production pushes a Prompt would remove.
- Development-selection is not a future holdout; market reaction is neither a causal nor a quality label.
- The required operator caveat register contains **{len(caveat_items)}** run-specific item(s); those limitations
  must travel with every executed report shared from this run.

### Recommended next experiment

{next_experiment}
'''
display(Markdown(takeaways))